# 03. Feature Engineering

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Where we are:** notebook 02 found three drivers of disruption (weather, geopolitical risk, carrier reliability) and one column that looks strong for a borrowed reason (lead time, which bad weather stretches).

**Goal of this notebook:** build a small set of features that

1. turn those findings into numbers a model can use,
2. mean something to a logistics coordinator, so the app can explain a prediction in plain words,
3. keep the model working when an input is missing,
4. use only what is known while the shipment is still moving.

We also test, and do not assume, whether the features improve a model.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA, OCHRE = "#2a78d6", "#eb6834", "#1baf7a", "#a07a12"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

## 1. Load the training file and lock away the outcome

Only one column is known after the fact: `Disruption_Occurred`. We separate it first so it cannot slip into a feature by accident.

In [2]:
clean = pd.read_csv(ROOT / "data" / "processed" / "aura_shipments.csv")

TARGET = "Disruption_Occurred"

inputs = clean.drop(columns=[TARGET])   # what a coordinator can see
y = clean[TARGET]                       # used only to score features

print(f"Shipments: {len(clean):,}   Disruption rate: {y.mean():.1%}")
print(f"Input columns ({inputs.shape[1]}):", list(inputs.columns))

Shipments: 5,000   Disruption rate: 61.3%
Input columns (22): ['Shipment_ID', 'Date', 'Place_of_Origin', 'Place_of_Destination', 'Distance_km', 'Transport_Mode', 'Lead_Time_Days', 'Transit_Duration_Hours', 'Stock_Volume', 'Geopolitical_Risk_Score', 'Fuel_Price_Index', 'Carrier_Reliability_Score', 'Weather_Condition', 'Product_Category', 'Buffer_of_Package', 'Storage_standard', 'Cargo_Value_USD', 'Hospital_Urgency_Tier', 'Flag_Land_Route_Across_Ocean', 'Flag_Lead_Time_At_Minimum', 'Flag_Speed_Impossible', 'Needs_Review']


## 2. The questions a coordinator asks

Every feature below answers one of these questions about a live shipment.

| Question | Columns involved | Used for |
|---|---|---|
| **How bad is the weather on the route?** | `Weather_Condition` | Predicting |
| **Is the trip already running slow?** | `Lead_Time_Days` against `Distance_km`, for the `Transport_Mode` | Predicting |
| **How many warning signs are stacked up?** | Weather, `Geopolitical_Risk_Score`, `Carrier_Reliability_Score` | Predicting, and as a rule of thumb |
| **If it is disrupted, how exposed is the cargo?** | `Buffer_of_Package` against the hours still to go | Ranking, not predicting |

The last question is a different kind. Notebook 02 showed that the cold-chain columns cannot predict a disruption. They describe how much damage one would do.

## 3. Build the features

Two of the features need a fixed reference.

- **Weather order.** We rank the five conditions by how much each slows a shipment down, measured in notebook 01 from lead times: clear, rain, fog, storm, hurricane. The order comes from the inputs, not from the outcome.
- **Typical pace.** The normal number of hours each mode takes per 1,000 km in clear weather. The next cell measures it.

In [3]:
pace = inputs["Lead_Time_Days"] * 24 / (inputs["Distance_km"] / 1000)
measured_pace = pace[inputs["Weather_Condition"] == "Clear"].groupby(inputs["Transport_Mode"]).median()

WEATHER_SEVERITY = {"Clear": 0, "Rain": 1, "Fog": 2, "Storm": 3, "Hurricane": 4}
TYPICAL_PACE = {"Air": 2.35, "Rail": 20.9, "Road": 17.9, "Sea": 41.5}   # hours per 1,000 km in clear weather

assert all(abs(measured_pace[m] - v) / v < 0.01 for m, v in TYPICAL_PACE.items()), "TYPICAL_PACE is out of date"
pd.DataFrame({"measured_in_clear_weather": measured_pace.round(2), "value_used": pd.Series(TYPICAL_PACE)})

,measured_in_clear_weather,value_used
Air,2.35,2.35
Rail,20.90,20.90
Road,17.90,17.90
Sea,41.53,41.50


In [4]:
HIGH_RISK_SCORE = 6.0        # geopolitical risk at or above this counts as a warning sign
LOW_RELIABILITY = 0.70       # carrier reliability below this counts as a warning sign


def add_features(df):
    """Add the engineered columns. Uses only inputs known during transit."""
    out = df.copy()
    severity = out["Weather_Condition"].map(WEATHER_SEVERITY)     # blank if the weather is missing or unknown
    hours = out["Lead_Time_Days"] * 24
    typical = out["Transport_Mode"].map(TYPICAL_PACE)

    # How bad is the weather?
    out["weather_severity"] = severity                            # 0 clear ... 4 hurricane
    out["severe_weather_flag"] = (severity >= 3).astype(float).where(severity.notna())

    # Is the trip already running slow?
    out["pace_h_per_1000km"] = hours / (out["Distance_km"] / 1000)
    out["pace_vs_typical"] = out["pace_h_per_1000km"] / typical    # 1.0 is normal, 3.0 is three times slower

    # How many warning signs are stacked up? (0 to 3)
    out["red_flag_count"] = (
        (severity >= 3).astype(int)
        + (out["Geopolitical_Risk_Score"] >= HIGH_RISK_SCORE).astype(int)
        + (out["Carrier_Reliability_Score"] < LOW_RELIABILITY).astype(int)
    )

    # If it is disrupted, how exposed is the cargo? (cold-chain only, blank for ambient cargo)
    out["hours_to_go"] = hours - out["Transit_Duration_Hours"]
    out["cold_gap_hours"] = out["Buffer_of_Package"] - out["hours_to_go"]   # below 0: buffer ends before arrival
    return out


ENGINEERED = ["weather_severity", "severe_weather_flag", "pace_h_per_1000km", "pace_vs_typical", "red_flag_count"]
EXPOSURE = ["hours_to_go", "cold_gap_hours"]

featured = add_features(inputs)
print("All predicting features are finite numbers:", bool(np.isfinite(featured[ENGINEERED]).all().all()))
featured[["Shipment_ID", "Weather_Condition", "Transport_Mode"] + ENGINEERED + EXPOSURE].head()

All predicting features are finite numbers: True


,Shipment_ID,Weather_Condition,Transport_Mode,weather_severity,severe_weather_flag,pace_h_per_1000km,pace_vs_typical,red_flag_count,hours_to_go,cold_gap_hours
0,SC-10000,Hurricane,Rail,4,1.0,167.490891,8.013918,1,240.56,NaN
1,SC-10001,Storm,Rail,3,1.0,68.747305,3.289345,3,517.48,NaN
2,SC-10002,Rain,Rail,1,0.0,24.920123,1.192350,1,47.16,-39.36
3,SC-10003,Hurricane,Rail,4,1.0,138.893639,6.645629,1,391.72,NaN
4,SC-10004,Fog,Air,2,0.0,4.344253,1.848618,0,9.80,12.00


### What each feature means

| Feature | Plain meaning | Formula | What a coordinator can do about it |
|---|---|---|---|
| `weather_severity` | How bad the weather is, 0 to 4 | clear 0, rain 1, fog 2, storm 3, hurricane 4 | Hold the shipment or route around the weather |
| `severe_weather_flag` | 1 if there is a storm or a hurricane | severity 3 or more | Same |
| `pace_h_per_1000km` | Hours the trip takes per 1,000 km | lead time in hours / (distance / 1,000) | Read only |
| `pace_vs_typical` | How many times slower than normal for this mode | pace / typical pace of the mode | Switch to a faster service |
| `red_flag_count` | Warning signs stacked up, 0 to 3 | severe weather + risk score 6 or more + reliability under 0.70 | Change carrier, change route |
| `hours_to_go` | Hours left until delivery | lead time in hours minus hours travelled | Read only |
| `cold_gap_hours` | Hours of cold left over on arrival | buffer minus hours to go | Replenish dry ice, speed up the transfer |

`red_flag_count` doubles as the rule of thumb a coordinator could apply without a model. Notebook 04 uses it as one of the baselines the model has to beat.

`cold_gap_hours` needs care. For air freight it is real: a passive box that runs out before arrival is in trouble. For sea, road and rail the cargo is in a powered container, and a negative gap only matters if the power fails. So it is shown to the coordinator as "thermal runway", and it is not given to the model.

## 4. How much signal does each feature carry?

We score every feature on its own against the outcome, using a number called **AUC**.

- Pick one disrupted shipment and one safe shipment at random. AUC is the chance that the feature ranks them the right way round.
- **0.50** is a coin flip (useless). **1.00** is perfect.

Engineered features are marked with a star. The two exposure features are scored on cold-chain shipments only, since they are blank for the rest.

In [5]:
from sklearn.metrics import roc_auc_score

RAW_NUMERIC = [
    "Geopolitical_Risk_Score", "Carrier_Reliability_Score", "Lead_Time_Days", "Distance_km",
    "Stock_Volume", "Fuel_Price_Index", "Transit_Duration_Hours", "Cargo_Value_USD", "Buffer_of_Package",
]


def strength(values, outcome):
    """AUC of one column against the outcome. Direction does not matter, so we fold it above 0.5."""
    known = values.notna()
    auc = roc_auc_score(outcome[known], values[known])
    return max(auc, 1 - auc)


signal = pd.Series({col: strength(featured[col], y) for col in ENGINEERED + EXPOSURE + RAW_NUMERIC}).sort_values()
kind = ["engineered" if c in ENGINEERED else "exposure" if c in EXPOSURE else "recorded or simulated" for c in signal.index]
labels = [f"{name} *" if name in ENGINEERED + EXPOSURE else name for name in signal.index]

fig = go.Figure(go.Bar(
    x=signal.values, y=labels, orientation="h",
    marker_color=[BLUE if k == "engineered" else AQUA if k == "exposure" else MUTED for k in kind],
    text=[f"{v:.2f}" for v in signal.values], textposition="outside", textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{y}<br>AUC: %{x:.3f}<extra></extra>",
))
fig.add_vline(x=0.5, line=dict(color=INK, width=1, dash="dash"))
fig.add_annotation(x=0.5, yref="paper", y=1.04, text="Coin flip", showarrow=False, xanchor="left", font=dict(color=INK))
fig.update_xaxes(range=[0.45, 0.85], title_text="AUC on its own")
style(fig, "Signal in each feature on its own (blue = engineered, green = exposure, grey = as given)", height=600)
fig.update_layout(margin=dict(l=220))
fig.show()

signal.sort_values(ascending=False).round(3).to_frame("auc_alone")

,auc_alone
weather_severity,0.775
pace_vs_typical,0.770
red_flag_count,0.739
severe_weather_flag,0.739
pace_h_per_1000km,0.662
Lead_Time_Days,0.640
Geopolitical_Risk_Score,0.633
Transit_Duration_Hours,0.633
hours_to_go,0.626
cold_gap_hours,0.612


**What this tells us:**

- **`weather_severity` and `pace_vs_typical` are the two strongest single signals**, both at about 0.77. They are close because they measure the same thing two ways: one reads the weather report, the other reads the slowdown that the weather causes.
- **`pace_vs_typical` is much stronger than the raw lead time** (0.77 against 0.64). Lead time mixes the weather effect with the mode and the distance. Dividing those out leaves the part that matters.
- **`red_flag_count` scores 0.74** using three yes/no questions. That is a strong rule of thumb, and a fair baseline for the model.
- **Geopolitical risk** scores 0.63 on its own and **carrier reliability** 0.55.
- **Distance, weight, fuel price, cargo value and buffer sit at 0.50.** They carry no signal about whether a disruption will happen.
- **Three columns score about 0.62 on borrowed signal.** `Transit_Duration_Hours`, `hours_to_go` and `cold_gap_hours` are all built from the lead time, so they pick up the weather effect that the lead time carries. They are weaker than the lead time itself and add nothing new. Section 8 checks this with a model.

## 5. Do the features behave sensibly?

A good feature should show risk rising steadily as the situation gets worse. Here is the disruption rate inside bands of the three main features.

In [6]:
panels = [
    ("weather_severity", [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], ["0 Clear", "1 Rain", "2 Fog", "3 Storm", "4 Hurricane"], "Weather severity"),
    ("pace_vs_typical", [0, 0.8, 1.25, 2, 4, 8, 999], ["Under 0.8x", "0.8 to 1.25x", "1.25 to 2x", "2 to 4x", "4 to 8x", "Over 8x"], "Pace against typical for the mode"),
    ("red_flag_count", [-0.5, 0.5, 1.5, 2.5, 3.5], ["0", "1", "2", "3"], "Red flag count"),
]

fig = make_subplots(rows=1, cols=3, subplot_titles=[p[3] for p in panels], horizontal_spacing=0.07)
band_tables = {}
for i, (col, edges, names, title) in enumerate(panels, start=1):
    bands = pd.cut(featured[col], bins=edges, labels=names)
    grouped = y.groupby(bands, observed=False).agg(["mean", "size"])
    band_tables[title] = grouped.rename(columns={"mean": "disruption_rate", "size": "shipments"})
    fig.add_trace(go.Bar(
        x=grouped.index.astype(str), y=grouped["mean"], marker_color=BLUE,
        text=[f"{m:.0%}" for m in grouped["mean"]], textposition="outside",
        textfont=dict(color=INK), cliponaxis=False, customdata=grouped["size"],
        hovertemplate="%{x}<br>Disrupted: %{y:.1%}<br>Shipments: %{customdata:,}<extra></extra>",
    ), row=1, col=i)
    fig.add_hline(y=y.mean(), line=dict(color=MUTED, width=1, dash="dash"), layer="below", row=1, col=i)
fig.update_annotations(font=dict(size=12, color=INK))
fig.update_yaxes(tickformat=".0%", range=[0, 1.15])
fig.update_yaxes(title_text="Disruption rate", col=1)
fig.update_xaxes(tickangle=-30)
style(fig, "Risk rises steadily as each feature gets worse (dashed line = all shipments)", height=460)
fig.update_layout(showlegend=False, bargap=0.3, margin=dict(b=100))
fig.show()

pd.concat(band_tables).round(3)

disruption_rate  shipments
Weather severity                  0 Clear                 0.370        995
                                  1 Rain                  0.420        977
                                  2 Fog                   0.481       1036
                                  3 Storm                 0.795       1002
                                  4 Hurricane             1.000        990
Pace against typical for the mode Under 0.8x              0.404        322
                                  0.8 to 1.25x            0.380       1348
                                  1.25 to 2x              0.511       1258
                                  2 to 4x                 0.757       1013
                                  4 to 8x                 0.943        651
                                  Over 8x                 0.973        408
Red flag count                    0                       0.289       1070
                                  1                       0.578       2153
                                  2                       0.824       1480
                                  3                       0.976        297

**What this tells us:** each feature moves in one direction, with no odd bumps.

- **Weather severity:** 37%, 42%, 48%, 80%, 100%. The order we fixed from lead times matches the order of risk.
- **Pace:** shipments moving at a normal pace are disrupted about 38% of the time. At four to eight times slower it is 94%.
- **Red flags:** 29% with none, 58% with one, 82% with two and 98% with all three. A coordinator can say this out loud: "three warning signs, almost certain trouble".

This is the behaviour the app needs to give a coordinator a believable explanation.

## 6. Are any features saying the same thing twice?

Two features that are almost identical add no information and make explanations confusing. We list the pairs whose correlation is above 0.80.

In [7]:
numeric_candidates = ENGINEERED + ["Geopolitical_Risk_Score", "Carrier_Reliability_Score", "Lead_Time_Days", "Distance_km"]
corr = featured[numeric_candidates].corr()

pairs = (
    corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    .stack().dropna().rename("correlation").reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
)
pairs = pairs[pairs["correlation"].abs() > 0.80].sort_values("correlation", key=abs, ascending=False)
print(f"Link between weather_severity and pace_vs_typical: {corr.loc['weather_severity', 'pace_vs_typical']:.2f}")
pairs.round(2).reset_index(drop=True)

Link between weather_severity and pace_vs_typical: 0.77


,feature_a,feature_b,correlation
0,weather_severity,severe_weather_flag,0.86
1,pace_h_per_1000km,Lead_Time_Days,0.83


**What this tells us:**

- Two pairs are above 0.80, and both are a feature next to the column it is built from: `weather_severity` with `severe_weather_flag`, and `pace_h_per_1000km` with `Lead_Time_Days`. We keep them. The number lets a model treat each step differently, and the flag is the plain yes/no a coordinator reads.
- `weather_severity` and `pace_vs_typical` are linked at 0.77. They overlap a great deal and are not copies. That is what we want from a backup: close enough to stand in, different enough to be a second reading.
- Tree-based models are not harmed by overlapping features. For a simple linear model, notebook 04 lets the tuning decide how much to lean on each.

## 7. Text columns

Models need numbers, so text columns are turned into yes/no columns, one per category. This is called one-hot encoding.

- `Weather_Condition` and `Transport_Mode` are encoded and given to the model. The mode does not predict disruption on its own, but the model needs it to read the pace correctly.
- `Product_Category` is tested below.
- `Shipment_ID`, the two ports and the date are labels, not causes. Notebook 01 showed the ports are not even tied to the distance. They are left out so the model cannot memorise individual shipments.
- `Storage_standard`, `Hospital_Urgency_Tier`, `Buffer_of_Package`, `Cargo_Value_USD` and `Transit_Duration_Hours` are simulated without looking at the outcome. They are tested below so that we check this and do not just assume it.

In [8]:
CORE_NUMERIC = ["Geopolitical_Risk_Score", "Carrier_Reliability_Score"]
TRIP_NUMERIC = ["Lead_Time_Days", "Distance_km"]
OTHER_NUMERIC = ["Stock_Volume", "Fuel_Price_Index"]
FLAGS = ["Flag_Land_Route_Across_Ocean", "Flag_Lead_Time_At_Minimum", "Flag_Speed_Impossible"]


def one_hot(columns):
    return pd.get_dummies(featured[columns]).astype(int)


weather = one_hot(["Weather_Condition"])
mode = one_hot(["Transport_Mode"])
product = one_hot(["Product_Category"])
cold_chain = pd.concat([
    one_hot(["Storage_standard", "Hospital_Urgency_Tier"]),
    featured[["Transit_Duration_Hours", "Cargo_Value_USD"] + FLAGS],
    featured["Buffer_of_Package"].fillna(0),   # blank for ambient cargo, which the storage columns already mark
], axis=1)
print("Encoded weather columns:", list(weather.columns))
print("Encoded mode columns   :", list(mode.columns))

Encoded weather columns: ['Weather_Condition_Clear', 'Weather_Condition_Fog', 'Weather_Condition_Hurricane', 'Weather_Condition_Rain', 'Weather_Condition_Storm']
Encoded mode columns   : ['Transport_Mode_Air', 'Transport_Mode_Rail', 'Transport_Mode_Road', 'Transport_Mode_Sea']


## 8. Do the features help a model?

Scoring features one at a time is not the whole story. Here we train two quick models on five different sets of columns and compare them.

**The five sets**
1. **Weather only:** what anyone gets from the weather report.
2. **Three core drivers:** weather, geopolitical risk and carrier reliability.
3. **All recorded inputs:** every recorded column, as given.
4. **Proposed:** the three core drivers, the trip columns and the five engineered features.
5. **Proposed plus everything else:** adds weight, fuel price, product type, the quality flags and all the simulated cold-chain columns.

**The two models**
- **Logistic regression:** a simple model that adds up weighted inputs.
- **Gradient boosting:** a stronger model built from many small decision trees.

**The two scores** (both averaged over 5 rounds, each round tested on shipments the model did not train on)
- **ROC AUC:** 0.50 is a coin flip, 1.00 is perfect.
- **Average precision:** how clean the top of the risk list is. A random list would score about 0.61, the disruption rate.

This is a quick comparison of feature sets, not model selection. Choosing and tuning the model is the job of notebook 04.

In [9]:
import warnings

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feature_sets = {
    "1. Weather only": weather,
    "2. Three core drivers": pd.concat([featured[CORE_NUMERIC], weather], axis=1),
    "3. All recorded inputs": pd.concat([featured[CORE_NUMERIC + TRIP_NUMERIC + OTHER_NUMERIC], weather, mode, product], axis=1),
    "4. Proposed": pd.concat([featured[CORE_NUMERIC + TRIP_NUMERIC + ENGINEERED], weather, mode], axis=1),
    "5. Proposed plus everything else": pd.concat(
        [featured[CORE_NUMERIC + TRIP_NUMERIC + OTHER_NUMERIC + ENGINEERED], weather, mode, product, cold_chain], axis=1),
}
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=42),
}
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def compare(sets):
    rows = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for set_name, X in sets.items():
            for model_name, model in models.items():
                scores = cross_validate(model, X, y, cv=folds, scoring=["roc_auc", "average_precision"])
                rows.append({
                    "feature_set": set_name, "model": model_name, "columns": X.shape[1],
                    "roc_auc": scores["test_roc_auc"].mean(), "roc_auc_spread": scores["test_roc_auc"].std(),
                    "avg_precision": scores["test_average_precision"].mean(),
                    "avg_precision_spread": scores["test_average_precision"].std(),
                })
    return pd.DataFrame(rows)


comparison = compare(feature_sets)
comparison.round(3)

,feature_set,model,columns,roc_auc,roc_auc_spread,avg_precision,avg_precision_spread
0,1. Weather only,Logistic regression,5,0.775,0.015,0.843,0.012
1,1. Weather only,Gradient boosting,5,0.775,0.015,0.843,0.012
2,2. Three core drivers,Logistic regression,7,0.831,0.013,0.901,0.008
3,2. Three core drivers,Gradient boosting,7,0.825,0.014,0.898,0.009
4,3. All recorded inputs,Logistic regression,20,0.829,0.013,0.900,0.009
5,3. All recorded inputs,Gradient boosting,20,0.822,0.012,0.896,0.008
6,4. Proposed,Logistic regression,18,0.830,0.012,0.901,0.008
7,4. Proposed,Gradient boosting,18,0.824,0.014,0.897,0.009
8,5. Proposed plus everything else,Logistic regression,39,0.829,0.012,0.900,0.008
9,5. Proposed plus everything else,Gradient boosting,39,0.823,0.011,0.896,0.007


In [10]:
def dot_chart(table, title, height=420):
    fig = make_subplots(rows=1, cols=2, subplot_titles=["ROC AUC", "Average precision"], horizontal_spacing=0.08,
                        shared_yaxes=True)
    model_colors = {"Logistic regression": BLUE, "Gradient boosting": ORANGE}
    for col, (metric, spread) in enumerate([("roc_auc", "roc_auc_spread"), ("avg_precision", "avg_precision_spread")], start=1):
        for model_name, color in model_colors.items():
            part = table[table["model"] == model_name]
            fig.add_trace(go.Scatter(
                x=part[metric], y=part["feature_set"], mode="markers", name=model_name,
                legendgroup=model_name, showlegend=col == 1,
                marker=dict(color=color, size=11, line=dict(color=SURFACE, width=2)),
                error_x=dict(type="data", array=part[spread], color=color, thickness=1.5, width=4),
                hovertemplate=model_name + "<br>%{y}<br>%{x:.3f}<extra></extra>",
            ), row=1, col=col)
    fig.update_yaxes(autorange="reversed")
    fig.update_annotations(font=dict(size=12, color=INK))
    style(fig, title, height=height)
    fig.update_layout(legend=dict(orientation="h", y=-0.2, x=0, font=dict(color=INK)), margin=dict(l=250, b=90))
    return fig


dot_chart(comparison, "Feature sets compared (dot = average of 5 rounds, bar = spread)").show()

**What this tells us, stated carefully:**

- **The big step is from one driver to three.** Weather alone gives a ROC AUC of about 0.78. Adding geopolitical risk and carrier reliability lifts it to about 0.83.
- **After that, nothing moves the score.** All recorded inputs, the proposed set and the everything set all land within the round-to-round spread of the three core drivers. The engineered features do not make the model more accurate, and we should not claim that they do.
- **The simple model is as good as the strong one.** Logistic regression matches or slightly beats gradient boosting on every set. The drivers add up in a plain way, with no hidden interactions for a complex model to find.
- **The simulated cold-chain columns add nothing**, which confirms what notebook 02 said about them.
- **About 0.83 looks like the ceiling for this file.** Thirty-nine columns do no better than seven. The rest of what decides a disruption is not recorded here.

So why keep the engineered features at all? The next section gives the reason.

## 9. What if the weather field is missing?

Weather is the strongest input. It is also the one most likely to arrive late, be left blank, or use a word the model has never seen. Here we take the weather column away completely and see what the trip columns can recover.

In [11]:
no_weather_sets = {
    "A. No weather, no trip columns": pd.concat([featured[CORE_NUMERIC], mode], axis=1),
    "B. No weather, raw lead time and distance": pd.concat([featured[CORE_NUMERIC + TRIP_NUMERIC], mode], axis=1),
    "C. No weather, with the pace features": pd.concat(
        [featured[CORE_NUMERIC + TRIP_NUMERIC + ["pace_h_per_1000km", "pace_vs_typical"]], mode], axis=1),
    "D. Weather present (proposed set)": feature_sets["4. Proposed"],
}
backup = compare(no_weather_sets)
dot_chart(backup, "How much of the weather signal the trip columns recover", height=380).show()
backup.round(3)

,feature_set,model,columns,roc_auc,roc_auc_spread,avg_precision,avg_precision_spread
0,"A. No weather, no trip columns",Logistic regression,6,0.640,0.003,0.722,0.010
1,"A. No weather, no trip columns",Gradient boosting,6,0.625,0.009,0.706,0.010
2,"B. No weather, raw lead time and distance",Logistic regression,8,0.766,0.014,0.850,0.012
3,"B. No weather, raw lead time and distance",Gradient boosting,8,0.798,0.018,0.878,0.011
4,"C. No weather, with the pace features",Logistic regression,10,0.805,0.016,0.880,0.011
5,"C. No weather, with the pace features",Gradient boosting,10,0.807,0.019,0.886,0.011
6,D. Weather present (proposed set),Logistic regression,18,0.830,0.012,0.901,0.008
7,D. Weather present (proposed set),Gradient boosting,18,0.824,0.014,0.897,0.009


**What this tells us:**

- **Without weather and without trip columns, the model loses most of its skill**: a ROC AUC of about 0.63.
- **The pace features recover most of the loss.** With them the model reaches about 0.81 with no weather field at all, against 0.83 with it.
- **The simple model needs the feature more than the strong one.** Given only the raw lead time and distance, gradient boosting works out the pace by itself and reaches about 0.80. Logistic regression cannot divide one column by another, so it stays at about 0.77 until we hand it `pace_vs_typical`.

This is the honest case for the engineered features. They do not raise the score when every field is present. They keep the model standing when the most important field is not, and they give the coordinator plain numbers to read: "storm, three times slower than normal, two red flags".

**One condition:** a model that has always been handed the weather never learns to lean on the pace. The backup needs its own model, trained without the weather columns. Notebook 05 builds it and measures the difference.

**A warning that goes with it:** the pace only works as a backup because `Lead_Time_Days` in this file already includes the weather delay. If the app is ever fed a *planned* lead time, set before the weather is known, the pace features will say "normal" for every shipment and the backup is gone.

## 10. What the shared code needs

The feature code in `src/features.py`, the column list in `src/schema.py` and the loader in `src/data_loader.py` were written for the previous training file. They still expect `Planned_Transit_Hours`, `Handover_Count` and a target called `Shipment_Failure`, none of which exist in this file, so the loader refuses it.

**Recommendation:** move `add_features` from this notebook into `src/features.py`, and update the column list and target name in `src/schema.py`, so that training, the app and the explanations all use the same names. That change touches shared files, so the team should agree on it first. Nothing in `src/` is changed by these notebooks.

## 11. Final check and feature list

In [12]:
MODEL_NUMERIC = CORE_NUMERIC + TRIP_NUMERIC + ENGINEERED
MODEL_CATEGORICAL = ["Weather_Condition", "Transport_Mode"]
MODEL_FEATURES = MODEL_NUMERIC + MODEL_CATEGORICAL
LEFT_OUT = [c for c in featured.columns if c not in MODEL_FEATURES]

assert TARGET not in MODEL_FEATURES, "The outcome is in the feature list"
assert not set(MODEL_FEATURES) & set(EXPOSURE), "An exposure feature is in the feature list"

model_table = featured[MODEL_FEATURES]
print(f"Outcome kept out of the feature list   : True")
print(f"Missing values in the feature table    : {int(model_table.isna().sum().sum())}")
print(f"Numeric features ({len(MODEL_NUMERIC)})  : {MODEL_NUMERIC}")
print(f"Text features ({len(MODEL_CATEGORICAL)})     : {MODEL_CATEGORICAL}")
print(f"Left out on purpose ({len(LEFT_OUT)}): {LEFT_OUT}")
model_table.describe().T.round(2)

Outcome kept out of the feature list   : True
Missing values in the feature table    : 0
Numeric features (9)  : ['Geopolitical_Risk_Score', 'Carrier_Reliability_Score', 'Lead_Time_Days', 'Distance_km', 'weather_severity', 'severe_weather_flag', 'pace_h_per_1000km', 'pace_vs_typical', 'red_flag_count']
Text features (2)     : ['Weather_Condition', 'Transport_Mode']
Left out on purpose (18): ['Shipment_ID', 'Date', 'Place_of_Origin', 'Place_of_Destination', 'Transit_Duration_Hours', 'Stock_Volume', 'Fuel_Price_Index', 'Product_Category', 'Buffer_of_Package', 'Storage_standard', 'Cargo_Value_USD', 'Hospital_Urgency_Tier', 'Flag_Land_Route_Across_Ocean', 'Flag_Lead_Time_At_Minimum', 'Flag_Speed_Impossible', 'Needs_Review', 'hours_to_go', 'cold_gap_hours']


,count,mean,std,min,25%,50%,75%,max
Geopolitical_Risk_Score,5000.0,5.08,2.88,0.00,2.60,5.10,7.50,10.00
Carrier_Reliability_Score,5000.0,0.75,0.14,0.50,0.63,0.76,0.88,1.00
Lead_Time_Days,5000.0,19.36,31.41,0.50,2.11,8.25,21.21,236.39
Distance_km,5000.0,7704.06,4199.69,500.17,4036.01,7750.12,11347.46,14995.91
weather_severity,5000.0,2.00,1.41,0.00,1.00,2.00,3.00,4.00
severe_weather_flag,5000.0,0.40,0.49,0.00,0.00,0.00,1.00,1.00
pace_h_per_1000km,5000.0,60.30,79.31,0.80,13.89,29.89,62.91,406.77
pace_vs_typical,5000.0,2.85,2.57,0.34,1.12,1.58,3.38,17.15
red_flag_count,5000.0,1.20,0.84,0.00,1.00,1.00,2.00,3.00


## What we found and what happens next

### Findings

| # | Finding | What we do about it |
|---|---|---|
| 1 | Three drivers carry the signal: weather, geopolitical risk, carrier reliability | They are the core of the model |
| 2 | `pace_vs_typical` turns lead time into a clean weather reading, 0.77 on its own against 0.64 | Use it in place of trusting raw lead time |
| 3 | `red_flag_count` scores 0.74 from three yes/no questions | Use it as the rule of thumb the model must beat |
| 4 | The engineered features do not raise the score when all fields are present | Report it honestly |
| 5 | With the weather field missing, the pace features hold the model at about 0.81 | Keep them as a backup |
| 6 | A simple model matches a complex one | Notebook 04 should compare both fairly |
| 7 | About 0.83 ROC AUC looks like the ceiling for this file | Do not promise more in the pitch |
| 8 | Weight, fuel price, product type and the simulated cold-chain columns add nothing | Leave them out of the model |
| 9 | Cargo value, urgency and `cold_gap_hours` describe what is at stake | Use them for ranking and money, not prediction |
| 10 | `src/` still expects the previous column names | Agree as a team, then update it |

### Next

Notebook 04 trains and compares models on `MODEL_FEATURES`, picks an alert line that keeps false alarms low, and reports precision and recall.

### Limits we state openly

- The file looks generated. Features built on the columns that drive its outcome are bound to score well. This shows the features recover the built-in pattern. It does not show they would work on a real carrier's records.
- The two cut-offs inside `red_flag_count` (risk score 6, reliability 0.70) are round numbers read off the band charts in notebook 02, which include every shipment. Notebook 04 scores the rule on held-out shipments only.
- The pace backup depends on the lead time including the weather delay, as explained in section 9.
- The feature comparison uses default model settings and one random split into 5 rounds.